# Predicting late shipments with a neural network

Same dataset and same split as `01_naive_bayes.ipynb`: E-Commerce Shipping Data, target
`Reached.on.Time_Y.N` (1 = did NOT arrive on time).

Naive Bayes assumes the features are independent given the class, which is almost never true. A
multilayer perceptron makes no such assumption, so it can pick up interactions between features.
This notebook builds one, tunes it, and reports train and test performance.

## Step 1: Load the data

Same file as the previous notebook.

In [1]:
import pandas as pd

df = pd.read_csv('Train.csv')

print(df.shape)
df.head()

(10999, 12)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


## Split before preprocessing

The outer test split remains 20%, stratified with seed 42. An inner validation split is made from raw training rows.
The search encoder and scaler are fitted only on inner training data. After selection, fresh preprocessing
is fitted on all outer training rows. Ten source features become 19 encoded inputs.


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=['ID', 'Reached.on.Time_Y.N'])
y = df['Reached.on.Time_Y.N']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_inner, X_valid, y_inner, y_valid = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
cat_cols = ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender']
num_cols = [c for c in X.columns if c not in cat_cols]
def make_preprocessor():
    return ColumnTransformer([
        ('numeric', StandardScaler(), num_cols),
        ('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)])
search_preprocessor = make_preprocessor()
X_tr = search_preprocessor.fit_transform(X_inner)
X_val = search_preprocessor.transform(X_valid)
y_tr, y_val = y_inner.to_numpy(), y_valid.to_numpy()
assert set(X_inner.index).isdisjoint(X_valid.index)
assert set(X_train.index).isdisjoint(X_test.index)
print('Inner training:', X_tr.shape, '| validation:', X_val.shape)


Inner training: (7039, 19) | validation: (1760, 19)


## Step 3: Build the network

The data is tabular, not images or sequences, so a plain feedforward MLP is the right shape.

`build_model()` is a function rather than a fixed model so Step 4 can call it repeatedly with
different hyperparameters.

- Hidden layers use `relu`.
- The output is a single neuron with `sigmoid`, since the target is binary. The output reads as
  the probability that the order is late.
- Loss is `binary_crossentropy`, optimiser is `Adam`.
- `l1_l2` regularisation on the hidden weights, to keep the network from memorising the training
  set.

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Fix random seeds to reduce run-to-run variation
SEED = 42
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

def build_model(input_dim, n_hidden_layers, units, reg_strength, lr=1e-3):
    reg = regularizers.l1_l2(l1=reg_strength, l2=reg_strength)

    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for _ in range(n_hidden_layers):
        model.add(layers.Dense(units, activation='relu', kernel_regularizer=reg))

    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


2026-09-18 14:43:17.866283: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Search eight configurations

Each model trains for the same 30 epochs. Selection uses validation accuracy at epoch 30, not the highest
score seen during training. Initial seeds are reset for each candidate. This is a small search on one split.


In [4]:
grid_hidden_layers = [1, 2]
grid_units = [8, 16]
grid_reg_strength = [1e-4, 1e-3]
EPOCHS, BATCH_SIZE = 30, 32
best_val_acc, best_params = -1.0, None
results = []
for n_layers in grid_hidden_layers:
    for units in grid_units:
        for strength in grid_reg_strength:
            keras.backend.clear_session()
            keras.utils.set_random_seed(SEED)
            model = build_model(X_tr.shape[1], n_layers, units, strength)
            history = model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                                epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
            score = history.history['val_accuracy'][-1]
            results.append((n_layers, units, strength, score))
            if score > best_val_acc:
                best_val_acc, best_params = score, (n_layers, units, strength)
            print(results[-1], flush=True)
print('Selected on validation:', best_params, '| accuracy:', best_val_acc)


(1, 8, 0.0001, 0.6579545736312866)


(1, 8, 0.001, 0.6625000238418579)


(1, 16, 0.0001, 0.6744318008422852)


(1, 16, 0.001, 0.6715909242630005)


(2, 8, 0.0001, 0.6659091114997864)


(2, 8, 0.001, 0.6670454740524292)


(2, 16, 0.0001, 0.6636363863945007)


(2, 16, 0.001, 0.6704545617103577)


Selected on validation: (1, 16, 0.0001) | accuracy: 0.6744318008422852


## Step 5: Train the final model

The winning combination is rebuilt and trained on the **full** training set, not just the reduced
portion used during the search, then evaluated on the training set and the held-out test set.

In [5]:
import json
from pathlib import Path

final_preprocessor = make_preprocessor()
X_train_final = final_preprocessor.fit_transform(X_train)
X_test_final = final_preprocessor.transform(X_test)
best_n_layers, best_units, best_reg_strength = best_params
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
final_model = build_model(X_train_final.shape[1], best_n_layers, best_units, best_reg_strength)
final_model.fit(X_train_final, y_train.to_numpy(), epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
train_loss, train_acc = final_model.evaluate(X_train_final, y_train.to_numpy(), verbose=0)
test_loss, test_acc = final_model.evaluate(X_test_final, y_test.to_numpy(), verbose=0)
probability = final_model.predict(X_test_final, verbose=0).ravel()
pd.DataFrame({'row_index': X_test.index, 'actual': y_test.to_numpy(), 'probability_late': probability}).to_csv('neural_network_predictions.csv', index=False)
Path('selected_model.json').write_text(json.dumps({
    'hidden_layers': best_n_layers, 'units': best_units, 'reg_strength': best_reg_strength,
    'epochs': EPOCHS, 'seed': SEED, 'validation_accuracy': float(best_val_acc),
    'test_accuracy': float(test_acc)}, indent=2) + '\n')
print('Train accuracy:', train_acc, '| test accuracy:', test_acc)


Train accuracy: 0.6815547347068787 | test accuracy: 0.6568182110786438


## Reading the result

The table below reports the current run. A small train-test gap alone does not establish the model's
capacity limit or prove that its features contain all available signal. The saved predictions are reused
by notebook 03, so that comparison measures this exact fitted model rather than another training run.


In [6]:
print(f'Train accuracy: {train_acc:.4f} | test accuracy: {test_acc:.4f}')
print(f'Train minus test: {(train_acc-test_acc)*100:.2f} percentage points')
print('Selected parameters:', best_params)


Train accuracy: 0.6816 | test accuracy: 0.6568
Train minus test: 2.47 percentage points
Selected parameters: (1, 16, 0.0001)


## Limits and provenance

This portfolio revision separates inner preprocessing from validation and saves final predictions.
The original course exercises used the same outer split; it has already been examined.
Feature availability at dispatch is not documented, particularly customer ratings and care calls.
Treat the scores as a retrospective classification exercise until those timestamps are established.
Seeds and deterministic operations improve repeatability in one environment; package and hardware changes can still affect results.
